# Track B — Head Grid v3 (TRACK_B_ARAHAN_V3.md)

**BDC Satria Data 2026** | Ganti head kNN -> MLP/logistic/LightGBM (A1), concat SigLIP2+SigLIP1 (A3),
bandingkan folds v1 vs v2 secara ADIL (A4/B3), pilih pemenang pakai aturan B9, serahkan ke Track C.

Notebook ini **tipis** -- semua logika ada di `track_b/src/` dan `track_b/experiments/*.py`
(sudah diuji CPU-only, lihat `track_b/tests/`). Sel di sini cuma memanggil fungsi yang sudah ada.

| | |
|---|---|
| Baseline yang harus dikalahkan | `siglip2so400m` + kNN, CV mean=0.9901 min=0.9896 std=0.0005, skor test 0.985 |
| Kandidat baru | `{siglip2so400m, concat_siglip2b256+siglip1b256_l2}` x `{linear, mlp, lgbm, ensemble}` |
| Aturan seleksi | B9 -- mean beda < 0.002 menang `min` tertinggi; mean naik tapi `min` turun = TOLAK |
| **Skor test 0.985** | **KONTEKS SAJA, bukan kriteria seleksi** -- tidak dipakai di sel mana pun di bawah |

---
## 🔧 SETUP

In [ ]:
# Cell 1 -- Repo + dependensi + Drive (CPU, tanpa GPU)
import os
if not os.path.exists('/content/satria-data-bdcugm02'):
    !git clone https://github.com/agaggigit/satria-data-bdcugm02.git
else:
    !git -C /content/satria-data-bdcugm02 pull

!pip install -q scikit-learn lightgbm

from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('setup siap')

In [ ]:
# Cell 2 -- sys.path ke src/ DAN experiments/ (head_grid_v3.py, fair_compare_v1_v2.py ada di sana)
import sys
sys.path.insert(0, '/content/satria-data-bdcugm02/track_b/src')
sys.path.insert(0, '/content/satria-data-bdcugm02/track_b/experiments')

from config import CFG
print('embeddings_dir:', CFG.embeddings_dir)
print('folds_csv     :', CFG.folds_csv)
print('folds_v2_csv  :', CFG.folds_v2_csv)
print('save_dir      :', CFG.save_dir)
print('seed          :', CFG.seed)

---
## ✅ VERIFIKASI ASET (Task 0 -- jangan lanjut kalau ada yang ❌)

Reuse `embed.available_embeddings` / `embed.verify_concat_dim` (sudah diuji `tests/test_embed.py`) --
bukan ditulis ulang di sini.

In [ ]:
# Cell 3 -- semua embedding yang tersedia, lalu gate keras: dim concat + aset wajib grid ini
from embed import EXPECTED_BACKBONES, available_embeddings, verify_concat_dim

for split in ('train', 'test'):
    found = available_embeddings(split, names=EXPECTED_BACKBONES)
    print(f'=== {split} ===')
    for name, info in found.items():
        print(f"  {name:16s} shape={info['shape']!s:16s} dim={info['dim']:5d} ckpt={info['checkpoint']}")
    missing = [n for n in EXPECTED_BACKBONES if n not in found]
    if missing:
        print(f'  belum ada: {missing}')

found_train = available_embeddings('train', names=EXPECTED_BACKBONES)
verify_concat_dim(found_train, 'siglip2b256', 'siglip1b256', expected=1536)
print('\nGATE A3 HIJAU -- dim concat 1536 terkonfirmasi')

assert 'siglip2so400m' in found_train, 'siglip2so400m belum di-cache -- combo ini wajib buat grid'
assert os.path.exists(CFG.folds_v2_csv), f'folds_v2.csv tidak ada: {CFG.folds_v2_csv}'
print('GATE ASET HIJAU -- siap jalankan grid')

---
## 🕸️ GRID -- head_grid_v3.py (Task 2)

`{siglip2so400m, concat_siglip2b256+siglip1b256_l2}` x `{linear, mlp, lgbm, ensemble}`,
dijalankan di `folds.csv` (v1) DAN `folds_v2.csv` (v2, angka NAIF). Resumable -- aman kalau sesi Colab putus.

In [ ]:
# Cell 4 -- jalankan grid penuh (fungsi sudah diuji, cuma dipanggil di sini)
import head_grid_v3
head_grid_v3.main()

---
## ⚖️ PERBANDINGAN ADIL v1 vs v2 -- fair_compare_v1_v2.py (Task 3, B3)

Angka `mean_naif_v2` dari sel di atas TIDAK sebanding v1 (eval set lebih mudah, sampel sulit sudah dibuang).
Sel ini menghitung `adil` (train v2, eval val-fold v1 UTUH) supaya bisa dibandingkan apple-to-apple.

In [ ]:
# Cell 5 -- baseline vs naif vs adil per (combo, head)
import fair_compare_v1_v2
fair_compare_v1_v2.main()

---
## 🏆 TABEL KEPUTUSAN + VERDICT (B9)

1. Pilih pemenang di `folds_version=1` pakai `selection.select_grid_winner` (mean beda < 0.002 -> `min` tertinggi menang).
2. Cek apakah pindah ke data v2 (ADIL, bukan naif) layak, pakai `selection.decide` -- aturan sama, cuma pairwise.
3. **Skor test 0.985 TIDAK muncul di sel mana pun di bawah** -- itu konteks, bukan kriteria (F: larangan keras).

In [ ]:
# Cell 6 -- tabel keputusan lengkap + pemenang + verdict v1-vs-v2
import pandas as pd
from selection import decide, select_grid_winner

REPO = '/content/satria-data-bdcugm02/track_b'
grid_df = pd.read_csv(f'{REPO}/results/head_grid_v3.csv')
fair_df = pd.read_csv(f'{REPO}/results/fair_compare_v1_v2.csv')

print('=== Tabel keputusan (folds_version=1) ===')
v1_df = grid_df[grid_df['folds_version'] == 1].sort_values('mean', ascending=False)
print(v1_df.to_string(index=False))

winner_v1 = select_grid_winner(v1_df)
print(f"\nPEMENANG (B9, folds_version=1): {winner_v1['combo']} + {winner_v1['head']} "
      f"(mean={winner_v1['mean']:.4f} min={winner_v1['min']:.4f} std={winner_v1['std']:.4f} "
      f"f1_electronic={winner_v1['f1_electronic']:.4f})")

match = fair_df[(fair_df['combo'] == winner_v1['combo']) & (fair_df['head'] == winner_v1['head'])]
assert len(match) == 1, (
    f"tidak ada baris fair_compare untuk pemenang {winner_v1['combo']}/{winner_v1['head']} -- "
    f"jalankan fair_compare_v1_v2.py utk kombinasi ini dulu"
)
fc = match.iloc[0]
print(f"\nfair_compare -- baseline_v1 mean={fc['mean_baseline_v1']:.4f} min={fc['min_baseline_v1']:.4f} "
      f"| adil_v2 mean={fc['mean_adil_v2']:.4f} min={fc['min_adil_v2']:.4f}")

verdict_v2 = decide(fc['mean_baseline_v1'], fc['min_baseline_v1'], fc['mean_adil_v2'], fc['min_adil_v2'])
print(f"Verdict pindah ke v2 (ADIL vs baseline v1): {verdict_v2}")

if verdict_v2 == 'TERIMA':
    FINAL_COMBO, FINAL_HEAD, FINAL_VERSION = winner_v1['combo'], winner_v1['head'], 2
    print(f"\n-> KOMPOSISI FINAL: {FINAL_COMBO} + {FINAL_HEAD}, folds_version=2 (OOF varian ADIL)")
else:
    FINAL_COMBO, FINAL_HEAD, FINAL_VERSION = winner_v1['combo'], winner_v1['head'], 1
    print(f"\n-> KOMPOSISI FINAL: {FINAL_COMBO} + {FINAL_HEAD}, folds_version=1 (tetap di v1)")

---
## 💾 HANDOFF ke Track C (guard anti-overwrite)

`handoff_v3.handoff_winner` menolak menimpa `oof.npy`/`oof_meta.json` yang sudah ada (mis. sisa era ConvNeXt/kNN)
kecuali `allow_overwrite=True` diset eksplisit. Kalau pemenang dari `folds_version=2`, otomatis diambil dari
varian **adil** (align ke `folds.csv` v1 penuh) -- BUKAN OOF naif v2 yang panjangnya beda.

In [ ]:
# Cell 7 -- tulis oof.npy + oof_meta.json kanonis, guard anti-overwrite
from handoff_v3 import handoff_winner

folds_v1_df = pd.read_csv(CFG.folds_csv)
CACHE_DIR = CFG.save_dir   # tempat head_grid_v3.py & fair_compare_v1_v2.py menulis oof_*.npy

meta = handoff_winner(
    cache_dir=CACHE_DIR, combo=FINAL_COMBO, head=FINAL_HEAD, folds_version=FINAL_VERSION,
    folds_v1=folds_v1_df, out_dir=CFG.save_dir, allow_overwrite=False,
)

print('HANDOFF SELESAI:')
for k, v in meta.items():
    print(f'  {k}: {v}')
print('\nUmumkan ke grup: "GATE 3 -> OOF head baru hijau, siap threshold tuning Track C"')

## Checklist & Langkah Berikutnya

- [ ] Cell 3: dim concat 1536 terkonfirmasi lewat kode (Task 0)
- [ ] Cell 4: grid lengkap, `results/head_grid_v3.csv` terisi (Task 2) -- per-class F1 Electronic ada di tiap baris
- [ ] Cell 5: perbandingan adil v1/v2 lengkap, `results/fair_compare_v1_v2.csv` terisi (Task 3)
- [ ] Cell 6: pemenang B9 terpilih, verdict v1-vs-v2 tercatat -- **bukan berdasar skor test 0.985**
- [ ] Cell 7: `oof.npy` + `oof_meta.json` tersimpan tanpa menimpa handoff lama secara diam-diam

**Kalau masih ada waktu GPU** setelah semua di atas hijau -> Task 5: `experiments/lora_ft.py`
(LoRA rank 8-16 di q_proj/v_proj, LR 1e-4; atau last-layer FT, LLRD). Smoke test fold 0 (2 epoch)
dulu sebelum full 5-fold -- lihat B8.